# Swimming Metrics

This notebook contains comprehensive swimming metrics calculations for elite swimmer performance analysis.

## Metrics Included:
- **WAP**: World Aquatic Points (1000 = World Record)
- **Stroke Rate**: Brazadas por minuto
- **DPS**: Distance Per Stroke (m/stroke)
- **RPE/sRPE**: Rate of Perceived Exertion
- **ACWR**: Acute-to-Chronic Workload Ratio

In [1]:
import numpy as np
import matplotlib.pyplot as plt
%load_ext autoreload
%autoreload 2

# Realistic Sample Data for Elite Swimmer

In [2]:
# Swimmer data

training_sessions = [
    {"distance": 100, "time": 48.5, "strokes": 38, "style": "free", "rpe": 7},
    {"distance": 200, "time": 108.2, "strokes": 82, "style": "free", "rpe": 8},
    {"distance": 400, "time": 238.5, "strokes": 220, "style": "free", "rpe": 7},
    {"distance": 800, "time": 502.3, "strokes": 480, "style": "free", "rpe": 6},
    {"distance": 1500, "time": 920.1, "strokes": 1050, "style": "free", "rpe": 5},
]

# Weekly training data (10 weeks) for ACWR
weekly_training = [
    {"volume_m": 5000, "time_min": 60, "rpe": 5},
    {"volume_m": 5500, "time_min": 60, "rpe": 6},
    {"volume_m": 6000, "time_min": 60, "rpe": 6},
    {"volume_m": 6500, "time_min": 60, "rpe": 7},
    {"volume_m": 7000, "time_min": 60, "rpe": 7},
    {"volume_m": 8000, "time_min": 60, "rpe": 8},
    {"volume_m": 7500, "time_min": 60, "rpe": 8},
    {"volume_m": 6500, "time_min": 60, "rpe": 6},
    {"volume_m": 5000, "time_min": 60, "rpe": 4},
    {"volume_m": 4000, "time_min": 60, "rpe": 3},
]

# Baseline (seconds)
world_records = {
    50:  {"free": 20.91, "back": 22.11, "breast": 25.95, "fly": 21.93},
    100: {"free": 44.84, "back": 49.48, "breast": 55.28, "fly": 49.45},
    200: {"free": 100.79, "back": 111.92, "breast": 124.18, "fly": 114.72},
    400: {"free": 220.07, "back": 237.76, "breast": 265.05, "fly": 236.32},
    800: {"free": 467.36},
    1500: {"free": 879.62},
}

print(f"Loaded {len(training_sessions)} training sessions and {len(weekly_training)} weeks of data")

Loaded 5 training sessions and 10 weeks of data


# World Aquatic Points (WAP)

**Formula:** $ Points = 1000 \cdot (B / T)^3 $

Where:
- B = Base time
- T = Swimmer's time.

In [3]:
def calculate_wap(base_time: float, swimmer_time: float) -> float:
    """Calculate World Aquatic Points."""
    return 1000 * (base_time / swimmer_time) ** 3

print("WAP Analysis:")
print("-" * 60)
for s in training_sessions:
    base = world_records.get(s["distance"], {}).get(s["style"], s["time"] * 1.05)
    wap = calculate_wap(base, s["time"])
    print(f"{s['distance']}m {s['style']}: Time={s['time']:.1f}s, Base={base:.1f}s, WAP={wap:.1f}")

WAP Analysis:
------------------------------------------------------------
100m free: Time=48.5s, Base=44.8s, WAP=790.3
200m free: Time=108.2s, Base=100.8s, WAP=808.3
400m free: Time=238.5s, Base=220.1s, WAP=785.6
800m free: Time=502.3s, Base=467.4s, WAP=805.5
1500m free: Time=920.1s, Base=879.6s, WAP=873.7


# Stroke Rate & Distance Per Stroke (DPS)

**Formulas:**
- $SR = Strokes / Time(minutes)$
- $DPS = Distance / Strokes$

In [4]:
def calculate_stroke_rate(strokes: int, time_sec: float) -> float:
    """Strokes per minute."""
    return strokes / (time_sec / 60)

def calculate_dps(distance: int, strokes: int) -> float:
    """Meters per stroke."""
    return distance / strokes if strokes > 0 else 0

print("Stroke Rate & DPS Analysis:")
print("-" * 70)
print(f"{'Distance':>8} | {'Strokes':>7} | {'Time':>6} | {'SR/min':>8} | {'DPS':>6} | {'Speed':>6}")
print("-" * 70)
for s in training_sessions:
    sr = calculate_stroke_rate(s["strokes"], s["time"])
    dps = calculate_dps(s["distance"], s["strokes"])
    speed = s["distance"] / s["time"]
    print(f"{s['distance']:>8}m | {s['strokes']:>7} | {s['time']:>6.1f}s | {sr:>8.1f} | {dps:>6.2f} | {speed:.2f}")

Stroke Rate & DPS Analysis:
----------------------------------------------------------------------
Distance | Strokes |   Time |   SR/min |    DPS |  Speed
----------------------------------------------------------------------
     100m |      38 |   48.5s |     47.0 |   2.63 | 2.06
     200m |      82 |  108.2s |     45.5 |   2.44 | 1.85
     400m |     220 |  238.5s |     55.3 |   1.82 | 1.68
     800m |     480 |  502.3s |     57.3 |   1.67 | 1.59
    1500m |    1050 |  920.1s |     68.5 |   1.43 | 1.63


# 4. Speed & Pace

- **Speed:** $Distance / Time$ (m/s)
- **Pace:** $Time / (Distance / 100)$ (s/100m)

In [5]:
def calculate_speed(distance: int, time_sec: float) -> float:
    """Speed in m/s."""
    return distance / time_sec

def calculate_pace(time_sec: float, distance: int) -> float:
    """Pace in seconds per 100m."""
    return time_sec / (distance / 100)

def pace_str(pace_sec: float) -> str:
    """Format pace as MM:SS."""
    return f"{int(pace_sec // 60)}:{int(pace_sec % 60):02d}"

print("Speed & Pace Analysis:")
print("-" * 50)
for s in training_sessions:
    speed = calculate_speed(s["distance"], s["time"])
    pace = calculate_pace(s["time"], s["distance"])
    print(f"{s['distance']}m: Speed={speed:.2f} m/s, Pace={pace_str(pace)}/100m")

Speed & Pace Analysis:
--------------------------------------------------
100m: Speed=2.06 m/s, Pace=0:48/100m
200m: Speed=1.85 m/s, Pace=0:54/100m
400m: Speed=1.68 m/s, Pace=0:59/100m
800m: Speed=1.59 m/s, Pace=1:02/100m
1500m: Speed=1.63 m/s, Pace=1:01/100m


# 6. RPE & sRPE

**Formula:** $ sRPE = RPE \times Duration(minutes) $

sRPE measures internal training load (subjective perception).

In [7]:
def calculate_wsrpe_min(rpe: float, duration_min: float) -> float:
    """Calculate Session RPE."""
    return rpe * duration_min

print("RPE & sRPE Analysis:")
print("-" * 65)
print(f"{'Week':>4} | {'Volume':>8} | {'Duration':>8} | {'RPE':>3} | {'sRPE':>5}")
print("-" * 65)
for i, w in enumerate(weekly_training, 1):
    wsrpe_min = calculate_wsrpe_min(w["rpe"], w["time_min"])
    print(f"{i:>4} | {w['volume_m']:>8}m | {w['time_min']:>8}min | {w['rpe']:>3} | {wsrpe_min:>5.0f}")

RPE & sRPE Analysis:
-----------------------------------------------------------------
Week |   Volume | Duration | RPE |  sRPE
-----------------------------------------------------------------
   1 |     5000m |       60min |   5 |   300
   2 |     5500m |       60min |   6 |   360
   3 |     6000m |       60min |   6 |   360
   4 |     6500m |       60min |   7 |   420
   5 |     7000m |       60min |   7 |   420
   6 |     8000m |       60min |   8 |   480
   7 |     7500m |       60min |   8 |   480
   8 |     6500m |       60min |   6 |   360
   9 |     5000m |       60min |   4 |   240
  10 |     4000m |       60min |   3 |   180


# 7. ACWR (Acute-to-Chronic Workload Ratio)

**Formula:** $ ACWR = Acute Load / Chronic Load $

**Safe range: 0.8 - 1.3**

Uses EWMA with decay factor $ \lambda = 2/(N+1) $ for chronic load.

$ EWMA_t = \alpha * r_t + (1 - \alpha) * EWMA_{t-1}$

In [33]:
import numpy as np

def numpy_ewma_vectorized(data, window: int):
    """
    Function
    """
    alpha = 2 /(window + 1.0)
    alpha_rev = 1-alpha

    scale = 1/alpha_rev
    n = data.shape[0]

    r = np.arange(n)
    scale_arr = scale**r
    offset = data[0]*alpha_rev**(r+1)
    pw0 = alpha*alpha_rev**(n-1)

    mult = data*pw0*scale_arr
    cumsums = mult.cumsum()
    out = offset + cumsums*scale_arr[::-1]
    return out

numpy_ewma_vectorized(np.array([1, 2, 3, 4, 5]), 4)

array([1.    , 1.4   , 2.04  , 2.824 , 3.6944])

In [28]:
def calculate_ewma(values: list, window: int, initial_ewma: float = 0.0) -> list:
    """
    Exponentially weighted moving average.

    EWMA_t = α * value_t + (1 - α) * EWMA_{t-1}

    where α = 1 / (window + 1) is the decay factor.

    Args:
        values (list): List of values.
        window (int): Window size.
        initial_ewma (float): Initial ewma value.

    Returns:
        list: List of ewma values for each timestep.
    """
    if not values:
        return []

    decay_factor = 1 / (window + 1)
    results = []
    ewma = initial_ewma

    for value in values:
        ewma = decay_factor * value + (1 - decay_factor) * ewma
        results.append(ewma)

    return results


# Ejemplo de uso
week = 6
window = 4
values = [1, 2, 3, 4, 3, 8]

# Slice: values[6-4-1 : 6-1] = values[1:5] = [2, 3, 4, 3]
result = calculate_ewma(values[week-window-1:week-1], window)

print(f"Window: {window}")
print(f"Values: {values[week-window-1:week-1]}")
print(f"EWMA sequence: {result}")
print(f"Final EWMA: {result[-1]:.6f}")

Window: 4
Values: [2, 3, 4, 3]
EWMA sequence: [0.4, 0.9200000000000002, 1.5360000000000003, 1.8288000000000004]
Final EWMA: 1.828800


In [29]:
sum([2, 3, 4, 3])/4

3.0

In [ ]:
def calculate_acwr(loads: list, week: int, window: int = 4) -> float:
    """
    Calculate ACWR with EWMA chronic load.

    Args:
        loads (list): List of loads.
        week (int): Week number.
        window (int): Window number.

    Returns:
        float: ACWR value.
    """
    if week < window:
        return -1
    chronic = calculate_ewma(loads[week-window-1:week], window)
    acute = loads[week]
    acwr = acute / chronic
    return acwr

srpe_loads = [calculate_wsrpe_min(w["rpe"], w["time_min"]) for w in weekly_training]
print("ACWR Analysis (sRPE-based Training Load):")
print("-" * 55)
print(f"{'Week':>4} | {'Chronic':>8} | {'ACWR':>5} | {'Status':<12}")
print("-" * 55)
for week in range(4, len(srpe_loads)):
    r = calculate_acwr(loads=srpe_loads, week=week, window=4)
    print(f"{week+1:>4} | {r['acute']:>6.0f} | {r['chronic']:>8.1f} | {r['acwr']:>5.2f} | {r['status']:<12}")

# TRIPM (Training Impulse)

